In [ ]:
%pip install --upgrade --force-reinstall numpy scipy scikit-learn neurokit2

In [ ]:
from pathlib import Path
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import mne
import neurokit2 as nk


# ============================================================
# PATHS
# ============================================================

REST_DIR = Path(r"D:\online_learning\resting_state")
TRAIN_DIR = Path(r"D:\online_learning\training_psychophysiology")

OUT_DIR = TRAIN_DIR / "neurokit_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_LONG = OUT_DIR / "physio_training_long_neurokit.csv"
OUT_QC = OUT_DIR / "physio_qc_neurokit.csv"


# ============================================================
# SETTINGS
# ============================================================

AUTO_FIX_VHDR_LINKS = True

GROUP1_IDS = set(list(range(1, 7)) + list(range(14, 21)) + [28, 29, 33])

# Resting baseline:
# resting trigger sequence may be S111 -> S222 -> S111 -> S222,
# but only two S111-locked resting segments are used.
REST_T0 = 10.0
REST_T1 = 100.0
REST_N_S111_SEGMENTS = 2

# Training:
# recording structure:
# training block 1 -> test block 1 -> training block 2 -> test block 2 -> training block 3
# training trigger = S111
# test trigger = S88
N_TRAINING_BLOCKS = 3
TRIALS_PER_TRAINING_BLOCK = 63
TRIAL_DURATION_SEC = 3.5

TRAIN_TRIGGER = "S111"
TEST_TRIGGER = "S88"

ECG_CANDIDATES = ["ECG", "EKG"]
EDA_CANDIDATES = ["EDA", "GSR", "EDR", "SCR"]


# ============================================================
# VHDR LINK CHECK / FIX
# ============================================================

def read_text_guess(path: Path):
    for enc in ["utf-8-sig", "utf-8", "latin-1", "cp1252"]:
        try:
            return path.read_text(encoding=enc), enc
        except UnicodeDecodeError:
            continue
    return path.read_text(errors="ignore"), "utf-8"


def linked_path_exists(vhdr_path: Path, linked_name: str) -> bool:
    if linked_name is None or str(linked_name).strip() == "":
        return False

    linked_name = str(linked_name).strip()
    p = Path(linked_name)

    if p.is_absolute():
        return p.exists()

    return (vhdr_path.parent / linked_name).exists()


def extract_vhdr_link(text: str, key: str):
    m = re.search(rf"^{key}=(.*)$", text, flags=re.MULTILINE)
    if m:
        return m.group(1).strip()
    return None


def find_expected_companion(vhdr_path: Path, suffix: str):
    exact = vhdr_path.with_suffix(suffix)
    if exact.exists():
        return exact.name

    folder = vhdr_path.parent
    stem_lower = vhdr_path.stem.lower()

    for p in folder.glob(f"*{suffix}"):
        if p.stem.lower() == stem_lower:
            return p.name

    return None


def fix_one_vhdr(vhdr_path: Path, verbose=True):
    text, enc = read_text_guess(vhdr_path)

    current_data = extract_vhdr_link(text, "DataFile")
    current_marker = extract_vhdr_link(text, "MarkerFile")

    data_exists = linked_path_exists(vhdr_path, current_data)
    marker_exists = linked_path_exists(vhdr_path, current_marker)

    expected_data = find_expected_companion(vhdr_path, ".eeg")
    expected_marker = find_expected_companion(vhdr_path, ".vmrk")

    new_text = text
    changed = False

    if not data_exists and expected_data is not None:
        if re.search(r"^DataFile=.*$", new_text, flags=re.MULTILINE):
            new_text = re.sub(
                r"^DataFile=.*$",
                f"DataFile={expected_data}",
                new_text,
                flags=re.MULTILINE
            )
        else:
            new_text += f"\nDataFile={expected_data}\n"
        changed = True

    if not marker_exists and expected_marker is not None:
        if re.search(r"^MarkerFile=.*$", new_text, flags=re.MULTILINE):
            new_text = re.sub(
                r"^MarkerFile=.*$",
                f"MarkerFile={expected_marker}",
                new_text,
                flags=re.MULTILINE
            )
        else:
            new_text += f"\nMarkerFile={expected_marker}\n"
        changed = True

    if changed:
        backup = vhdr_path.with_suffix(vhdr_path.suffix + ".bak")
        if not backup.exists():
            shutil.copy2(vhdr_path, backup)

        vhdr_path.write_text(new_text, encoding=enc)

        if verbose:
            print(f"[FIXED] {vhdr_path.name}")
            print(f"  Old DataFile   = {current_data} | exists: {data_exists}")
            print(f"  Old MarkerFile = {current_marker} | exists: {marker_exists}")
            print(f"  New DataFile   = {expected_data if not data_exists else current_data}")
            print(f"  New MarkerFile = {expected_marker if not marker_exists else current_marker}")

    else:
        if verbose:
            status = "OK" if data_exists and marker_exists else "NOT FIXED"
            print(f"[{status}] {vhdr_path.name}")
            print(f"  DataFile   = {current_data} | exists: {data_exists}")
            print(f"  MarkerFile = {current_marker} | exists: {marker_exists}")

    return changed


def check_and_fix_vhdr_folder(folder: Path):
    print(f"\nChecking BrainVision links in: {folder}")

    for vhdr in sorted(folder.glob("*.vhdr")):
        if AUTO_FIX_VHDR_LINKS:
            fix_one_vhdr(vhdr, verbose=True)
        else:
            text, _ = read_text_guess(vhdr)
            data_file = extract_vhdr_link(text, "DataFile")
            marker_file = extract_vhdr_link(text, "MarkerFile")

            print(f"\n{vhdr.name}")
            print(f"  DataFile   = {data_file} | exists: {linked_path_exists(vhdr, data_file)}")
            print(f"  MarkerFile = {marker_file} | exists: {linked_path_exists(vhdr, marker_file)}")


# ============================================================
# GENERAL HELPERS
# ============================================================

def parse_subject_id(path: Path) -> int:
    """
    Example:
    OL04_rest.vhdr -> 04
    OL04_train.vhdr -> 04
    """
    name = path.name

    if len(name) >= 4:
        two = re.sub(r"\D", "", name[2:4])
        if len(two) == 2:
            return int(two)

    m = re.search(r"\d{2}", name)
    if m:
        return int(m.group())

    raise ValueError(f"Cannot parse subject ID from filename: {path.name}")


def normalize_trigger(desc: str) -> str:
    """
    Examples:
    Stimulus/S111 -> S111
    S 111 -> S111
    Stimulus/S 88 -> S88
    """
    s = str(desc).upper().strip()
    s = s.replace("STIMULUS/", "")
    s = s.replace("RESPONSE/", "")
    s = re.sub(r"\s+", "", s)
    return s


def load_raw_vhdr(path: Path):
    raw = mne.io.read_raw_brainvision(
        path,
        preload=True,
        misc="auto",
        verbose=False
    )
    return raw


def get_events(raw):
    events = []

    for onset, desc in zip(raw.annotations.onset, raw.annotations.description):
        events.append((float(onset), normalize_trigger(desc)))

    return events


def find_channel(raw, candidates):
    names = raw.ch_names
    upper_names = [x.upper() for x in names]

    for cand in candidates:
        cand = cand.upper()
        for i, name in enumerate(upper_names):
            if cand in name:
                return names[i]

    return None


def extract_segment(raw, channel_name, start_sec, end_sec):
    if channel_name is None:
        return None

    fs = float(raw.info["sfreq"])

    start = int(round(start_sec * fs))
    end = int(round(end_sec * fs))

    data = raw.copy().pick([channel_name]).get_data()[0]

    start = max(start, 0)
    end = min(end, len(data))

    if end <= start:
        return None

    return data[start:end]


# ============================================================
# RESTING BASELINE EXTRACTION
# ============================================================

def select_rest_triggers(events):
    """
    Resting baseline rule:

    Actual resting trigger sequence may be:
    S111 -> S222 -> S111 -> S222

    Analysis rule:
    only use the first two S111 triggers.

    Extract:
    1. first S111 + 10–100 s
    2. second S111 + 10–100 s

    S222 triggers are ignored.
    """
    s111_events = [(onset, label) for onset, label in events if label == "S111"]

    if len(s111_events) < REST_N_S111_SEGMENTS:
        raise RuntimeError(
            f"Found only {len(s111_events)} S111 triggers for resting baseline; "
            f"need {REST_N_S111_SEGMENTS}."
        )

    return s111_events[:REST_N_S111_SEGMENTS]


# ============================================================
# NEUROKIT2 ECG / EDA PROCESSING
# ============================================================

def process_ecg_segment(ecg, fs):
    out = {
        "HR": np.nan,
        "RMSSD": np.nan,
        "ECG_Quality": np.nan,
        "N_RPeaks": 0,
        "ECG_OK": False,
        "ECG_Error": "",
    }

    if ecg is None or len(ecg) < int(10 * fs):
        out["ECG_Error"] = "missing_or_too_short"
        return out

    try:
        signals, info = nk.ecg_process(
            ecg,
            sampling_rate=fs,
            method="neurokit"
        )

        rpeaks = info.get("ECG_R_Peaks", [])
        out["N_RPeaks"] = int(len(rpeaks))

        if "ECG_Rate" in signals:
            out["HR"] = float(np.nanmean(signals["ECG_Rate"]))

        if "ECG_Quality" in signals:
            out["ECG_Quality"] = float(np.nanmean(signals["ECG_Quality"]))

        if len(rpeaks) >= 5:
            hrv = nk.hrv_time(rpeaks, sampling_rate=fs, show=False)
            if "HRV_RMSSD" in hrv:
                out["RMSSD"] = float(hrv["HRV_RMSSD"].iloc[0])

        out["ECG_OK"] = bool(np.isfinite(out["HR"]) and np.isfinite(out["RMSSD"]))

    except Exception as e:
        out["ECG_Error"] = str(e)

    return out


def process_eda_segment(eda, fs):
    out = {
        "SCL_mean": np.nan,
        "SCR_rate": np.nan,
        "SCR_amp_mean": np.nan,
        "N_SCR": 0,
        "EDA_OK": False,
        "EDA_Error": "",
    }

    if eda is None or len(eda) < int(10 * fs):
        out["EDA_Error"] = "missing_or_too_short"
        return out

    try:
        # amplitude_min=0.1: candidate peak prominence / maximum candidate
        # prominence within THIS segment >= 0.1 (not SCR_Amplitude in conductance units).
        signals, info = nk.eda_process(
            eda, sampling_rate=fs, method="neurokit", amplitude_min=0.1
        )

        if "EDA_Tonic" in signals:
            out["SCL_mean"] = float(np.nanmean(signals["EDA_Tonic"]))

        duration_min = len(eda) / fs / 60.0

        scr_peaks = info.get("SCR_Peaks", [])
        out["N_SCR"] = int(len(scr_peaks))
        out["SCR_rate"] = float(out["N_SCR"] / duration_min) if duration_min > 0 else np.nan

        if "SCR_Amplitude" in info and len(info["SCR_Amplitude"]) > 0:
            out["SCR_amp_mean"] = float(np.nanmean(info["SCR_Amplitude"]))
        elif "SCR_Amplitude" in signals.columns and "SCR_Peaks" in signals.columns:
            amp = signals.loc[signals["SCR_Peaks"] == 1, "SCR_Amplitude"]
            out["SCR_amp_mean"] = float(np.nanmean(amp)) if len(amp) > 0 else 0.0
        else:
            out["SCR_amp_mean"] = np.nan

        out["EDA_OK"] = bool(np.isfinite(out["SCL_mean"]) and np.isfinite(out["SCR_rate"]))

    except Exception as e:
        out["EDA_Error"] = str(e)

    return out


def average_metric_dicts(dicts, keys):
    if len(dicts) == 0:
        return {k: np.nan for k in keys}

    df = pd.DataFrame(dicts)

    out = {}
    for k in keys:
        if k in df.columns:
            out[k] = float(df[k].mean(skipna=True))
        else:
            out[k] = np.nan

    return out


def extract_rest_baseline(raw):
    """
    Extract two 90-s resting baseline segments:

    first S111 + 10–100 s
    second S111 + 10–100 s

    Each segment is processed separately.
    Segment-level estimates are then averaged.
    """
    fs = float(raw.info["sfreq"])
    events = get_events(raw)

    ecg_ch = find_channel(raw, ECG_CANDIDATES)
    eda_ch = find_channel(raw, EDA_CANDIDATES)

    selected = select_rest_triggers(events)

    ecg_metrics = []
    eda_metrics = []
    qc_rows = []

    for i, (onset, label) in enumerate(selected, start=1):
        start = onset + REST_T0
        end = onset + REST_T1

        ecg_seg = extract_segment(raw, ecg_ch, start, end)
        eda_seg = extract_segment(raw, eda_ch, start, end)

        ecg_out = process_ecg_segment(ecg_seg, fs) if ecg_ch is not None else {
            "HR": np.nan,
            "RMSSD": np.nan,
            "ECG_Quality": np.nan,
            "N_RPeaks": 0,
            "ECG_OK": False,
            "ECG_Error": "no_ecg_channel",
        }

        eda_out = process_eda_segment(eda_seg, fs) if eda_ch is not None else {
            "SCL_mean": np.nan,
            "SCR_rate": np.nan,
            "SCR_amp_mean": np.nan,
            "N_SCR": 0,
            "EDA_OK": False,
            "EDA_Error": "no_eda_channel",
        }

        ecg_metrics.append(ecg_out)
        eda_metrics.append(eda_out)

        qc_rows.append({
            "SegmentType": "rest",
            "Segment": f"R{i}",
            "Trigger": label,
            "Start_sec": start,
            "End_sec": end,
            "Duration_sec": end - start,
            "ECG_Channel": ecg_ch,
            "EDA_Channel": eda_ch,
            **{f"ECG_{k}": v for k, v in ecg_out.items()},
            **{f"EDA_{k}": v for k, v in eda_out.items()},
        })

    ecg_base = average_metric_dicts(
        ecg_metrics,
        ["HR", "RMSSD", "ECG_Quality", "N_RPeaks"]
    )

    eda_base = average_metric_dicts(
        eda_metrics,
        ["SCL_mean", "SCR_rate", "SCR_amp_mean", "N_SCR"]
    )

    return ecg_base, eda_base, qc_rows


# ============================================================
# TRAINING BLOCK EXTRACTION
# ============================================================

def define_training_blocks_by_test_periods(events):
    """
    Define three training blocks when each test block contains many S88 triggers.

    Recording structure:
        training 1 -> test 1 -> training 2 -> test 2 -> training 3

    Triggers:
        S111 = training trial
        S88  = test trial

    Important:
        A test block contains many S88 triggers. Therefore, we do NOT use
        the first two S88 events as the two test boundaries. Instead, we detect
        the start of each test period as the first S88 after a run of S111 trials.

    Block definitions:
        B1 = the last 63 S111 triggers before the start of test period 1
        B2 = the last 63 S111 triggers before the start of test period 2
        B3 = the last 63 S111 triggers after test period 2, i.e., final training period
    """

    blocks = []
    current_s111 = []
    in_test_period = False

    for onset, label in events:

        if label == TRAIN_TRIGGER:
            # If we were in a test period, this S111 marks the start of
            # the next training period.
            if in_test_period:
                in_test_period = False
                current_s111 = []

            current_s111.append(onset)

        elif label == TEST_TRIGGER:
            # This is the first S88 after a training run.
            # Therefore, it marks the start of a test period.
            if not in_test_period and len(current_s111) > 0:

                # Only the first two test-period starts define B1 and B2.
                if len(blocks) < 2:
                    if len(current_s111) < TRIALS_PER_TRAINING_BLOCK:
                        raise RuntimeError(
                            f"Before test period {len(blocks) + 1}, found only "
                            f"{len(current_s111)} S111 triggers; "
                            f"need {TRIALS_PER_TRAINING_BLOCK}."
                        )

                    block_s111 = current_s111[-TRIALS_PER_TRAINING_BLOCK:]

                    blocks.append({
                        "block_name": f"B{len(blocks) + 1}",
                        "s111_onsets": block_s111,
                        "boundary_s88_onset": onset,
                        "definition": (
                            f"last {TRIALS_PER_TRAINING_BLOCK} S111 before "
                            f"test period {len(blocks) + 1}"
                        ),
                    })

                # Enter test period.
                # All following S88 triggers are ignored until S111 appears again.
                in_test_period = True
                current_s111 = []

            else:
                # Repeated S88 triggers inside the same test period are ignored.
                continue

        else:
            # Other triggers are ignored.
            continue

    # After the second test period, the final training period should remain
    # in current_s111. B3 is the final 63 S111 triggers from this final run.
    if len(blocks) < 2:
        raise RuntimeError(
            f"Expected two test periods, but detected only {len(blocks)}."
        )

    if len(current_s111) < TRIALS_PER_TRAINING_BLOCK:
        raise RuntimeError(
            f"For Block 3, found only {len(current_s111)} S111 triggers "
            f"after the second test period; need {TRIALS_PER_TRAINING_BLOCK}."
        )

    block3_s111 = current_s111[-TRIALS_PER_TRAINING_BLOCK:]

    blocks.append({
        "block_name": "B3",
        "s111_onsets": block3_s111,
        "boundary_s88_onset": None,
        "definition": f"final {TRIALS_PER_TRAINING_BLOCK} S111 after second test period",
    })

    if len(blocks) != N_TRAINING_BLOCKS:
        raise RuntimeError(
            f"Expected {N_TRAINING_BLOCKS} training blocks, found {len(blocks)}."
        )

    return blocks


def extract_training_blocks(raw):
    """
    Extract three listen-and-repeat training blocks.

    B1:
        last 63 S111 triggers before the start of test period 1.

    B2:
        last 63 S111 triggers before the start of test period 2.

    B3:
        final 63 S111 triggers after test period 2.

    For B1 and B2, the end of the segment is capped at the first S88 of the
    following test period, so that the test block is not included.
    """
    fs = float(raw.info["sfreq"])
    events = get_events(raw)

    ecg_ch = find_channel(raw, ECG_CANDIDATES)
    eda_ch = find_channel(raw, EDA_CANDIDATES)

    training_blocks = define_training_blocks_by_test_periods(events)

    rows = []
    qc_rows = []

    for b, block_info in enumerate(training_blocks, start=1):
        s111_onsets = block_info["s111_onsets"]
        boundary_s88_onset = block_info["boundary_s88_onset"]

        if len(s111_onsets) != TRIALS_PER_TRAINING_BLOCK:
            raise RuntimeError(
                f"Block {b}: expected {TRIALS_PER_TRAINING_BLOCK} S111 triggers, "
                f"but found {len(s111_onsets)}."
            )

        first_s111 = s111_onsets[0]
        last_s111 = s111_onsets[-1]

        start = first_s111
        end_candidate = last_s111 + TRIAL_DURATION_SEC

        # B1 and B2 should not include the following S88-marked test period.
        if boundary_s88_onset is not None:
            end = min(end_candidate, boundary_s88_onset)
        else:
            end = end_candidate

        ecg_seg = extract_segment(raw, ecg_ch, start, end)
        eda_seg = extract_segment(raw, eda_ch, start, end)

        ecg_out = process_ecg_segment(ecg_seg, fs) if ecg_ch is not None else {
            "HR": np.nan,
            "RMSSD": np.nan,
            "ECG_Quality": np.nan,
            "N_RPeaks": 0,
            "ECG_OK": False,
            "ECG_Error": "no_ecg_channel",
        }

        eda_out = process_eda_segment(eda_seg, fs) if eda_ch is not None else {
            "SCL_mean": np.nan,
            "SCR_rate": np.nan,
            "SCR_amp_mean": np.nan,
            "N_SCR": 0,
            "EDA_OK": False,
            "EDA_Error": "no_eda_channel",
        }

        row = {
            "Block": f"B{b}",
            "Block_start_sec": start,
            "Block_end_sec": end,
            "Block_duration_sec": end - start,
            "N_S111_in_block": len(s111_onsets),
            "Boundary_S88_onset_sec": boundary_s88_onset,
            "Block_definition": block_info["definition"],

            "HR": ecg_out.get("HR", np.nan),
            "RMSSD": ecg_out.get("RMSSD", np.nan),
            "ECG_Quality": ecg_out.get("ECG_Quality", np.nan),
            "N_RPeaks": ecg_out.get("N_RPeaks", np.nan),

            "SCL_mean": eda_out.get("SCL_mean", np.nan),
            "SCR_rate": eda_out.get("SCR_rate", np.nan),
            "SCR_amp_mean": eda_out.get("SCR_amp_mean", np.nan),
            "N_SCR": eda_out.get("N_SCR", np.nan),
        }

        rows.append(row)

        qc_rows.append({
            "SegmentType": "training",
            "Segment": f"B{b}",
            "Trigger": TRAIN_TRIGGER,
            "Start_sec": start,
            "End_sec": end,
            "Duration_sec": end - start,
            "N_S111_in_block": len(s111_onsets),
            "Boundary_S88_onset_sec": boundary_s88_onset,
            "Block_definition": block_info["definition"],
            "ECG_Channel": ecg_ch,
            "EDA_Channel": eda_ch,
            **{f"ECG_{k}": v for k, v in ecg_out.items()},
            **{f"EDA_{k}": v for k, v in eda_out.items()},
        })

    return rows, qc_rows


# ============================================================
# FILE MAPS
# ============================================================

def build_file_maps():
    rest_files = sorted([
        p for p in REST_DIR.glob("*.vhdr")
        if "_rest" in p.name.lower()
    ])

    train_files = sorted([
        p for p in TRAIN_DIR.glob("*.vhdr")
        if "_rest" not in p.name.lower()
    ])

    rest_map = {}
    train_map = {}

    for p in rest_files:
        try:
            sid = parse_subject_id(p)
            rest_map.setdefault(sid, p)
        except Exception as e:
            warnings.warn(f"Skipping rest file {p.name}: {e}")

    for p in train_files:
        try:
            sid = parse_subject_id(p)
            train_map.setdefault(sid, p)
        except Exception as e:
            warnings.warn(f"Skipping train file {p.name}: {e}")

    return rest_map, train_map


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline():
    check_and_fix_vhdr_folder(REST_DIR)
    check_and_fix_vhdr_folder(TRAIN_DIR)

    rest_map, train_map = build_file_maps()

    all_rows = []
    all_qc = []

    print("\n============================================================")
    print("Start NeuroKit2 ECG/EDA extraction")
    print("============================================================")

    for sid in sorted(rest_map.keys()):
        if sid not in train_map:
            warnings.warn(f"Subject {sid:02d}: missing training file, skipped.")
            continue

        group = "Group1" if sid in GROUP1_IDS else "Group2"

        print(f"\nProcessing subject {sid:02d} | {group}")
        print(f"  Rest : {rest_map[sid].name}")
        print(f"  Train: {train_map[sid].name}")

        try:
            raw_rest = load_raw_vhdr(rest_map[sid])
            ecg_base, eda_base, rest_qc = extract_rest_baseline(raw_rest)

            raw_train = load_raw_vhdr(train_map[sid])
            block_rows, train_qc = extract_training_blocks(raw_train)

            for q in rest_qc + train_qc:
                q["Subject"] = sid
                q["Group"] = group
                all_qc.append(q)

            for block in block_rows:
                all_rows.append({
                    "Subject": sid,
                    "Group": group,
                    "Block": block["Block"],

                    "HR": block["HR"],
                    "RMSSD": block["RMSSD"],
                    "ECG_Quality": block["ECG_Quality"],
                    "N_RPeaks": block["N_RPeaks"],

                    "SCL_mean": block["SCL_mean"],
                    "SCR_rate": block["SCR_rate"],
                    "SCR_amp_mean": block["SCR_amp_mean"],
                    "N_SCR": block["N_SCR"],

                    "HR_baseline": ecg_base.get("HR", np.nan),
                    "RMSSD_baseline": ecg_base.get("RMSSD", np.nan),
                    "ECG_Quality_baseline": ecg_base.get("ECG_Quality", np.nan),
                    "N_RPeaks_baseline": ecg_base.get("N_RPeaks", np.nan),

                    "SCL_baseline": eda_base.get("SCL_mean", np.nan),
                    "SCR_baseline": eda_base.get("SCR_rate", np.nan),
                    "SCR_amp_baseline": eda_base.get("SCR_amp_mean", np.nan),
                    "N_SCR_baseline": eda_base.get("N_SCR", np.nan),

                    "Block_start_sec": block["Block_start_sec"],
                    "Block_end_sec": block["Block_end_sec"],
                    "Block_duration_sec": block["Block_duration_sec"],
                    "N_S111_in_block": block["N_S111_in_block"],
                    "Boundary_S88_onset_sec": block["Boundary_S88_onset_sec"],
                    "Block_definition": block["Block_definition"],
                })

        except Exception as e:
            warnings.warn(f"Subject {sid:02d} failed: {e}")
            all_qc.append({
                "Subject": sid,
                "Group": group,
                "SegmentType": "subject",
                "Segment": "all",
                "Error": str(e),
            })

    df = pd.DataFrame(all_rows)
    qc = pd.DataFrame(all_qc)

    df.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")
    qc.to_csv(OUT_QC, index=False, encoding="utf-8-sig")

    print("\n============================================================")
    print("DONE")
    print("============================================================")
    print(f"Saved long table:\n  {OUT_LONG}")
    print(f"Saved QC table:\n  {OUT_QC}")

    if not df.empty:
        print("\nRows by Group x Block:")
        print(df.groupby(["Group", "Block"]).size())

        print("\nS111 count by block:")
        print(df.groupby("Block")["N_S111_in_block"].describe())

        print("\nMissing summary:")
        cols = [
            "HR", "RMSSD", "SCL_mean", "SCR_rate",
            "HR_baseline", "RMSSD_baseline", "SCL_baseline", "SCR_baseline"
        ]
        print(df[cols].isna().sum())

        print("\nSegmentation preview:")
        print(df[[
            "Subject",
            "Group",
            "Block",
            "N_S111_in_block",
            "Block_start_sec",
            "Block_end_sec",
            "Block_duration_sec",
            "Boundary_S88_onset_sec",
            "Block_definition"
        ]].head(30))

    return df, qc


# ============================================================
# RUN
# ============================================================

df, qc = run_pipeline()

In [ ]:
import numpy as np
import pandas as pd

# qc 是 run_pipeline() 输出的 QC table
qc_check = qc.copy()

# ------------------------------------------------------------
# Basic duration checks
# ------------------------------------------------------------

# baseline should be 90 s
qc_check["Flag_bad_rest_duration"] = (
    (qc_check["SegmentType"] == "rest") &
    (np.abs(qc_check["Duration_sec"] - 90.0) > 1.0)
)

# training block duration should roughly match 63 trials * 3.5 s
# 允许一定误差，因为 trigger timing 可能有 jitter
expected_train_duration = 63 * 3.5
qc_check["Flag_bad_train_duration"] = (
    (qc_check["SegmentType"] == "training") &
    (np.abs(qc_check["Duration_sec"] - expected_train_duration) > 10.0)
)

# ------------------------------------------------------------
# ECG QC
# ------------------------------------------------------------

# Dynamic minimum R-peak count based on very conservative HR = 40 bpm
qc_check["Expected_min_RPeaks"] = qc_check["Duration_sec"] * 40 / 60

qc_check["Flag_ecg_missing"] = qc_check["ECG_ECG_OK"].astype(str).str.lower().isin(["false", "nan"])

qc_check["Flag_low_rpeaks"] = (
    pd.to_numeric(qc_check["ECG_N_RPeaks"], errors="coerce") <
    qc_check["Expected_min_RPeaks"]
)

qc_check["Flag_hr_outlier"] = (
    (pd.to_numeric(qc_check["ECG_HR"], errors="coerce") < 40) |
    (pd.to_numeric(qc_check["ECG_HR"], errors="coerce") > 180)
)

qc_check["Flag_rmssd_outlier"] = (
    (pd.to_numeric(qc_check["ECG_RMSSD"], errors="coerce") <= 0) |
    (pd.to_numeric(qc_check["ECG_RMSSD"], errors="coerce") > 300)
)

# ECG_Quality threshold is not universal.
# This is a practical flag, not automatic exclusion.
qc_check["Flag_low_ecg_quality"] = (
    pd.to_numeric(qc_check["ECG_ECG_Quality"], errors="coerce") < 0.5
)

# ------------------------------------------------------------
# EDA QC
# ------------------------------------------------------------

qc_check["Flag_eda_missing"] = qc_check["EDA_EDA_OK"].astype(str).str.lower().isin(["false", "nan"])

qc_check["Flag_scl_invalid"] = ~np.isfinite(
    pd.to_numeric(qc_check["EDA_SCL_mean"], errors="coerce")
)

qc_check["Flag_scr_rate_extreme"] = (
    pd.to_numeric(qc_check["EDA_SCR_rate"], errors="coerce") > 30
)

qc_check["Flag_scr_invalid"] = ~np.isfinite(
    pd.to_numeric(qc_check["EDA_SCR_rate"], errors="coerce")
)

# ------------------------------------------------------------
# Combined flag
# ------------------------------------------------------------

flag_cols = [
    "Flag_bad_rest_duration",
    "Flag_bad_train_duration",
    "Flag_ecg_missing",
    "Flag_low_rpeaks",
    "Flag_hr_outlier",
    "Flag_rmssd_outlier",
    "Flag_low_ecg_quality",
    "Flag_eda_missing",
    "Flag_scl_invalid",
    "Flag_scr_rate_extreme",
    "Flag_scr_invalid",
]

qc_check["Any_QC_Flag"] = qc_check[flag_cols].any(axis=1)

# Save full QC table
qc_out = r"D:\online_learning\training_psychophysiology\neurokit_out\physio_qc_flags.csv"
qc_check.to_csv(qc_out, index=False, encoding="utf-8-sig")

# Print flagged segments
flagged = qc_check[qc_check["Any_QC_Flag"]].copy()

print("Number of flagged segments:", len(flagged))
display(flagged[[
    "Subject", "Group", "SegmentType", "Segment", "Duration_sec",
    "ECG_HR", "ECG_RMSSD", "ECG_ECG_Quality", "ECG_N_RPeaks",
    "EDA_SCL_mean", "EDA_SCR_rate", "EDA_N_SCR",
    *flag_cols
]])

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

base_dir = Path(r"D:\online_learning\training_psychophysiology\neurokit_out")

long_csv = base_dir / "physio_training_long_neurokit.csv"
qc_csv = base_dir / "physio_qc_flags.csv"

out_csv = base_dir / "physio_training_long_neurokit_scr_cleaned.csv"
log_csv = base_dir / "scr_extreme_cleaning_log.csv"


# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(long_csv)
qc = pd.read_csv(qc_csv)


# ============================================================
# HELPER
# ============================================================

def to_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes", "y"]


qc["Flag_scr_rate_extreme_bool"] = (
    qc["Flag_scr_rate_extreme"].apply(to_bool) |
    ~np.isfinite(pd.to_numeric(qc["EDA_SCR_rate"], errors="coerce"))
)
# SCR exclusion is metric-specific: retain finite SCL in the same segment.
# These are automatic criteria; this script does not establish visual review.

log_rows = []


# ============================================================
# 1. TRAINING BLOCKS:
#    If training block SCR_rate is extreme,
#    set SCR-related training values to NaN only.
# ============================================================

qc_train_bad = qc[
    (qc["SegmentType"].astype(str).str.lower() == "training") &
    (qc["Flag_scr_rate_extreme_bool"])
].copy()

for _, row in qc_train_bad.iterrows():
    sid = row["Subject"]
    block = row["Segment"]   # should be B1, B2, or B3

    mask = (df["Subject"] == sid) & (df["Block"].astype(str) == str(block))

    n_changed = int(mask.sum())

    if n_changed > 0:
        old_values = df.loc[mask, ["SCR_rate", "SCR_amp_mean", "N_SCR"]].copy()

        df.loc[mask, ["SCR_rate", "SCR_amp_mean", "N_SCR"]] = np.nan

        log_rows.append({
            "Action": "training_SCR_set_to_NaN",
            "Subject": sid,
            "Block_or_Segment": block,
            "N_rows_changed": n_changed,
            "Old_SCR_rate": old_values["SCR_rate"].tolist(),
            "Old_SCR_amp_mean": old_values["SCR_amp_mean"].tolist(),
            "Old_N_SCR": old_values["N_SCR"].tolist(),
            "Reason": "Flag_scr_rate_extreme=True in training block"
        })
    else:
        log_rows.append({
            "Action": "training_no_matching_row_found",
            "Subject": sid,
            "Block_or_Segment": block,
            "N_rows_changed": 0,
            "Old_SCR_rate": "",
            "Old_SCR_amp_mean": "",
            "Old_N_SCR": "",
            "Reason": "Flag_scr_rate_extreme=True but no matching Subject+Block row in long table"
        })


# ============================================================
# 2. RESTING BASELINE:
#    If one rest segment has extreme SCR_rate,
#    recompute SCR baseline using only valid rest segments.
#    If all rest segments are bad, set SCR baseline to NaN.
# ============================================================

qc_rest = qc[
    qc["SegmentType"].astype(str).str.lower() == "rest"
].copy()

subjects = sorted(qc_rest["Subject"].dropna().unique())

for sid in subjects:
    rest_sub = qc_rest[qc_rest["Subject"] == sid].copy()

    if rest_sub.empty:
        continue

    # valid rest segments for SCR baseline:
    # not Flag_scr_rate_extreme and finite EDA_SCR_rate
    valid = rest_sub[
        (~rest_sub["Flag_scr_rate_extreme_bool"]) &
        (np.isfinite(pd.to_numeric(rest_sub["EDA_SCR_rate"], errors="coerce")))
    ].copy()

    mask_subject = df["Subject"] == sid

    if mask_subject.sum() == 0:
        continue

    old_baseline = df.loc[
        mask_subject,
        ["SCR_baseline", "SCR_amp_baseline", "N_SCR_baseline"]
    ].drop_duplicates()

    # Only update subjects who had at least one rest SCR extreme flag
    had_bad_rest_scr = rest_sub["Flag_scr_rate_extreme_bool"].any()

    if not had_bad_rest_scr:
        continue

    if len(valid) > 0:
        new_scr_baseline = pd.to_numeric(valid["EDA_SCR_rate"], errors="coerce").mean()

        if "EDA_SCR_amp_mean" in valid.columns:
            new_scr_amp_baseline = pd.to_numeric(valid["EDA_SCR_amp_mean"], errors="coerce").mean()
        else:
            new_scr_amp_baseline = np.nan

        if "EDA_N_SCR" in valid.columns:
            new_n_scr_baseline = pd.to_numeric(valid["EDA_N_SCR"], errors="coerce").mean()
        else:
            new_n_scr_baseline = np.nan

        df.loc[mask_subject, "SCR_baseline"] = new_scr_baseline
        df.loc[mask_subject, "SCR_amp_baseline"] = new_scr_amp_baseline
        df.loc[mask_subject, "N_SCR_baseline"] = new_n_scr_baseline

        log_rows.append({
            "Action": "rest_SCR_baseline_recomputed_from_valid_segments",
            "Subject": sid,
            "Block_or_Segment": ",".join(valid["Segment"].astype(str).tolist()),
            "N_rows_changed": int(mask_subject.sum()),
            "Old_SCR_rate": old_baseline.to_dict("records"),
            "Old_SCR_amp_mean": "",
            "Old_N_SCR": "",
            "Reason": "At least one rest segment had Flag_scr_rate_extreme=True; baseline recomputed from valid rest segments"
        })

    else:
        df.loc[mask_subject, ["SCR_baseline", "SCR_amp_baseline", "N_SCR_baseline"]] = np.nan

        log_rows.append({
            "Action": "rest_SCR_baseline_set_to_NaN",
            "Subject": sid,
            "Block_or_Segment": ",".join(rest_sub["Segment"].astype(str).tolist()),
            "N_rows_changed": int(mask_subject.sum()),
            "Old_SCR_rate": old_baseline.to_dict("records"),
            "Old_SCR_amp_mean": "",
            "Old_N_SCR": "",
            "Reason": "All rest segments invalid for SCR baseline because Flag_scr_rate_extreme=True or missing EDA_SCR_rate"
        })


# ============================================================
# SAVE
# ============================================================
# Handle non-finite SCL separately and recompute its resting baseline from
# valid SCL segments; never discard SCL solely because SCR rate is excessive.
df["SCL_mean"] = pd.to_numeric(df["SCL_mean"], errors="coerce").replace([np.inf, -np.inf], np.nan)
for sid, resting in qc_rest.groupby("Subject"):
    scl = pd.to_numeric(resting["EDA_SCL_mean"], errors="coerce")
    scl = scl[np.isfinite(scl)]
    df.loc[df["Subject"] == sid, "SCL_baseline"] = scl.mean() if len(scl) else np.nan


log_df = pd.DataFrame(log_rows, columns=["Action", "Subject", "Block_or_Segment", "N_rows_changed", "Old_SCR_rate", "Old_SCR_amp_mean", "Old_N_SCR", "Reason"])

df.to_csv(out_csv, index=False, encoding="utf-8-sig")
log_df.to_csv(log_csv, index=False, encoding="utf-8-sig")

print("Saved cleaned long table:")
print(out_csv)

print("\nSaved cleaning log:")
print(log_csv)

print("\nSummary:")
print("Training blocks with SCR_rate set to NaN:", len(qc_train_bad))
print("Subjects with rest SCR baseline updated:", log_df["Action"].astype(str).str.contains("rest_SCR").sum())

print("\nCleaning actions:")
print(log_df["Action"].value_counts())

## Heart rate / RMSSD distribution and LMM diagnostics
Run the following cell independently using existing CSV exports. It compares raw and natural-log scales and fits diagnostic random-intercept models. Recommendations do not change the existing MATLAB inference. Review the saved plots before writing a claim about residual adequacy.

In [ ]:
# Run independently of raw ECG/EDA extraction using the exported analysis table.
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

base = Path(r"D:\online_learning\training_psychophysiology\neurokit_out")
source = base / "physio_training_long_neurokit_scr_cleaned.csv"
if not source.exists():
    source = base / "physio_training_long_neurokit.csv"
data = pd.read_csv(source)
out = base / "ecg_distribution_diagnostics"
out.mkdir(parents=True, exist_ok=True)
# Transparent exploratory rule, not a normality requirement for LMM outcomes.
# Recommend ln only when raw positive skew > 1 and ln reduces absolute skew.
# Fit BOTH scales; final inference requires review of residuals and scientific context.
summary, residual_summary, notes = [], [], []
models = {}
for outcome in ["HR", "RMSSD"]:
    baseline = outcome + "_baseline"
    d = data[["Subject", "Group", "Block", outcome, baseline]].copy()
    for col in [outcome, baseline]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    if d.duplicated(["Subject", "Block"]).any():
        raise ValueError("Duplicate Subject/Block rows in analysis table")
    if len(d) < 8 or d.Subject.nunique() < 3:
        raise ValueError(f"Insufficient complete observations for {outcome}")
    positive = bool((d[[outcome, baseline]] > 0).all().all())
    raw_skew = stats.skew(d[outcome], bias=False)
    log_skew = stats.skew(np.log(d[outcome]), bias=False) if positive else np.nan
    recommend_log = positive and raw_skew > 1 and abs(log_skew) < abs(raw_skew)
    notes.append(f"{outcome}: n={len(d)}, subjects={d.Subject.nunique()}, raw skew={raw_skew:.3f}, ln skew={log_skew:.3f}; recommended scale={'ln' if recommend_log else 'raw'}.")
    for scale in (["raw", "ln"] if positive else ["raw"]):
        work = d.copy()
        work["response"] = np.log(d[outcome]) if scale == "ln" else d[outcome]
        work["baseline_covariate"] = np.log(d[baseline]) if scale == "ln" else d[baseline]
        x = work.response.to_numpy()
        summary.append(dict(outcome=outcome, scale=scale, n=len(x), subjects=d.Subject.nunique(), excluded_rows=len(data)-len(d), skew=stats.skew(x, bias=False), excess_kurtosis=stats.kurtosis(x, bias=False), shapiro_p=stats.shapiro(x).pvalue, recommended=(scale == ("ln" if recommend_log else "raw"))))
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
        axes[0].hist(x, bins="auto", edgecolor="black")
        axes[0].set_title(f"{outcome} ({scale}) distribution")
        stats.probplot(x, dist="norm", plot=axes[1])
        fig.tight_layout()
        fig.savefig(out / f"{outcome}_{scale}_distribution.png", dpi=160)
        plt.close(fig)
        # Diagnostic random-intercept model matching existing fixed effects.
        # This does not reproduce MATLAB random-slope selection or Type-III tests.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = smf.mixedlm("response ~ C(Group)*C(Block) + baseline_covariate", work, groups=work.Subject)
            fit = model.fit(reml=True, method=["lbfgs", "bfgs", "powell"])
        models[(outcome, scale)] = fit
        (out / f"{outcome}_{scale}_model.txt").write_text(str(fit.summary()) + "\nWarnings:\n" + "\n".join(str(w.message) for w in caught), encoding="utf-8")
        try:
            fitted = np.asarray(fit.fittedvalues)
            residual = np.asarray(fit.resid)
        except (ValueError, np.linalg.LinAlgError) as exc:
            notes.append(f"{outcome} {scale}: conditional residuals unavailable ({exc}); inspect model singularity.")
            continue
        residual_summary.append(dict(outcome=outcome, scale=scale, converged=fit.converged, residual_skew=stats.skew(residual, bias=False), residual_shapiro_p=stats.shapiro(residual).pvalue, random_intercept_variance=float(fit.cov_re.iloc[0,0]), warnings="; ".join(str(w.message) for w in caught)))
        pd.DataFrame({"Subject":work.Subject, "Group":work.Group, "Block":work.Block, "fitted":fitted, "residual":residual}).to_csv(out / f"{outcome}_{scale}_residuals.csv", index=False)
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
        stats.probplot(residual, dist="norm", plot=axes[0])
        axes[1].scatter(fitted, residual, alpha=.65)
        axes[1].axhline(0, color="black", linestyle="--")
        axes[1].set(xlabel="Conditional fitted values", ylabel="Conditional residuals")
        axes[2].scatter(fitted, np.sqrt(np.abs(residual / np.std(residual, ddof=1))), alpha=.65)
        axes[2].set(xlabel="Conditional fitted values", ylabel="sqrt(|standardized residual|)")
        fig.suptitle(f"{outcome} ({scale}): random-intercept model diagnostics")
        fig.tight_layout()
        fig.savefig(out / f"{outcome}_{scale}_residual_diagnostics.png", dpi=160)
        plt.close(fig)

pd.DataFrame(summary).to_csv(out / "distribution_summary.csv", index=False)
pd.DataFrame(residual_summary).to_csv(out / "residual_summary.csv", index=False)
notes += ["Shapiro p-values are descriptive: repeated observations are dependent; marginal normality is not an LMM assumption.", "Inspect residual Q-Q, residual-versus-fitted, and scale-location plots before claiming adequacy. Convergence does not establish adequacy.", "Recommendations are exploratory. Existing MATLAB analyses remain on the original scale; a recommended transformation has NOT been applied to those analyses.", "For ln-scale models, both the outcome and its corresponding baseline are natural-log transformed; no offset or automatic ECG outlier exclusion is used.", f"Input: {source}"]
report = "\n".join(notes)
(out / "diagnostic_report.txt").write_text(report, encoding="utf-8")
print(report)
print(pd.DataFrame(summary).to_string(index=False))
print(pd.DataFrame(residual_summary).to_string(index=False))


### Preliminary diagnostic results
The preceding diagnostic fits compare scales only. The following MATLAB cell performs the final original-HR / ln-RMSSD analysis, including natural-log baseline RMSSD, ML random-effects comparison, REML estimation and Satterthwaite Type-III tests. Use its outputs for inference.


In [ ]:
# Run this cell independently after CSV export; MATLAB Statistics Toolbox required.
from pathlib import Path
import subprocess
matlab = Path(r"S:\Program\matlab2019a\bin\matlab.exe")
analysis_script = Path(r"D:\online_learning\training_psychophysiology\script\other\ECG_LMM_logRMSSD_TypeIII.m")
completed = subprocess.run([str(matlab), "-wait", "-nosplash", "-nodesktop", "-batch", "run('" + analysis_script.as_posix() + "')"], capture_output=True, text=True)
print(completed.stdout)
if completed.returncode:
    raise RuntimeError(completed.stderr or completed.stdout)
results = analysis_script.parents[2] / "neurokit_out" / "lmm_stats_from_neurokit_logRMSSD"
print((results / "ECG_LMM_TypeIII_summary.txt").read_text(encoding="utf-8"))


## Final transformed analysis results (2026-10-04)

Input: cleaned NeuroKit CSV; 32 subjects, 96 observations (Group1 n=15, Group2 n=17). HR analysed in bpm; RMSSD and its resting baseline natural-log transformed. ML comparison retained random intercepts for HR (p=.05805) and selected categorical Block random slopes for ln RMSSD (p=.000002914). Final fits use REML; fixed-effect tests use effects coding and Satterthwaite df.

|Outcome|Effect|F|df1|df2|p|
|---|---|---:|---:|---:|---:|
|HR|Group|3.6594|1|29|.065676|
|HR|Block|2.9675|2|60|.059026|
|HR|Group x Block|.79006|2|60|.45848|
|HR|Resting HR baseline|179.31|1|29|6.0378e-14|
|ln RMSSD|Group|.75145|1|29.007|.39313|
|ln RMSSD|Block|1.2600|2|30|.29823|
|ln RMSSD|Group x Block|.76858|2|30|.47259|
|ln RMSSD|ln resting RMSSD baseline|49.446|1|29|9.8363e-8|

No Group, Block or interaction test reached .05. This does not establish equivalence or absence of an effect. Baseline covariates were positively associated with training outcomes.

Final conditional residual skewness: HR=.530303; ln RMSSD=.605508. Both final Q-Q plots show tail departures. The earlier ln RMSSD residual skewness of 2.916 refers to the preliminary random-intercept fit, not the selected random-slope model. Do not claim assumptions were fully confirmed. The categorical random-slope model is complex relative to three observations per subject; its covariance structure and sensitivity to a simpler model merit review before publication.

Methods wording:

> The distributions of heart rate and RMSSD were examined using histograms and normal Q–Q plots before statistical modelling. RMSSD exhibited substantial positive skewness and was therefore natural-log transformed, whereas heart rate was analysed on its original scale. The corresponding resting-state RMSSD baseline covariate was also natural-log transformed. Conditional residuals from the fitted linear mixed-effects models were examined using normal Q–Q plots, residual-versus-fitted plots, and scale–location plots to assess normality and homoscedasticity.

Results wording:

> After adjustment for resting-state baseline values, neither the group effect, block effect, nor group-by-block interaction reached statistical significance for heart rate or log-transformed RMSSD (all p > .05). Resting-state baseline values were positively associated with both outcomes. Residual diagnostics indicated departures in the tails of the distributions, which should be considered when interpreting the model-based inference.

Full MATLAB output and final residual plots: `neurokit_out/lmm_stats_from_neurokit_logRMSSD/`. The Python random-intercept diagnostics remain supplementary and are not the source of the reported F tests.


In [ ]:
# EDA distribution/QC audit and final original-SCL / ln-SCR-rate LMM.
# Run independently using existing CSV exports. No visual-review claim is implied.
from pathlib import Path
import runpy
import subprocess
script_dir = Path(r"D:\online_learning\training_psychophysiology\script")
runpy.run_path(str(script_dir / "EDA_distribution_QC_audit.py"), run_name="__main__")
matlab = Path(r"S:\Program\matlab2019a\bin\matlab.exe")
eda_script = script_dir / "other" / "EDA_LMM_logSCRrate_TypeIII.m"
result = subprocess.run([str(matlab), "-wait", "-nosplash", "-nodesktop", "-batch", "run('" + eda_script.as_posix() + "')"], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or result.stdout)
results_dir = script_dir.parent / "neurokit_out" / "lmm_stats_from_neurokit_logSCRrate"
print((results_dir / "EDA_LMM_TypeIII_summary.txt").read_text(encoding="utf-8", errors="replace"))


## EDA processing, quality control and final analysis

EDA processing and distribution checks

EDA signals were processed using NeuroKit2's neurokit pipeline, including fourth-order Butterworth low-pass filtering at 3 Hz. Tonic and phasic activity were obtained using low- and high-pass filters, respectively, with a cutoff frequency of 0.05 Hz. Candidate SCR peaks were detected in the phasic component using a relative threshold of 0.1. Specifically, each candidate peak's prominence was divided by the maximum prominence among candidate peaks within the same independently processed segment, and peaks with a ratio of at least 0.1 were retained. Thus, the threshold was dimensionless and did not represent an absolute conductance amplitude or the subsequently calculated onset-to-peak SCR amplitude. Mean SCL was calculated as the average tonic component, and SCR rate as the number of detected peaks divided by segment duration in minutes, for each resting-state segment and training block.

The distributions of mean SCL and SCR rate were examined using histograms and normal Q-Q plots before statistical modelling. SCL showed mild positive skewness (0.72) and was analysed on its original scale. SCR rate exhibited pronounced positive skewness (2.88) and was natural-log transformed; its corresponding resting-state baseline was also natural-log transformed. Among the 93 observations included in the SCR model, skewness decreased from 2.88 to -0.33 following transformation. Conditional residuals were examined using normal Q-Q plots, residual-versus-fitted plots, and scale-location plots to assess normality and homoscedasticity.

Quality control and missing data

ECG segments were screened for R-peak counts below those expected at 40 beats/min, mean heart rate outside 40-180 beats/min, non-positive RMSSD or RMSSD above 300 ms, and ECG quality indices below .50. No ECG segments met these ECG-specific criteria, and no resting-state segments or training-block observations were excluded for ECG. Both ECG models included 96 observations from 32 participants.

EDA segments were screened for non-finite SCL or SCR-rate estimates and SCR rates exceeding 30 responses/min. The existing cleaning script treated SCR-related estimates from segments exceeding this rate criterion as missing; finite SCL estimates from the same segments were retained. No SCL observations were excluded. For SCR, three resting-state segments from two participants were excluded, whereas no training-block SCR observations were directly excluded. For one participant, the SCR baseline was calculated from the remaining valid resting-state segment. For the other participant, both resting-state segments were invalid and the SCR baseline was treated as missing. Consequently, that participant's three training observations could not enter the baseline-adjusted SCR model. The SCL model included 96 observations from 32 participants, and the SCR model included 93 observations from 31 participants. Models used all observations with available outcome and baseline values, without imputation of missing model variables.

Visual inspection is not documented in the existing QC logs. Therefore, the statement 'Flagged segments were visually inspected' must only be added if the investigator confirms that this was actually done. These exclusions were SCR-specific; do not describe them as exclusions of all EDA measures. A separate duration flag was present for one training segment, but it was not an exclusion under the documented cleaning rule.

Statistical results

Models included Group, Block, Group x Block and the corresponding resting baseline. Random effects were compared under ML; final fits used REML and Type-III Satterthwaite tests. SCL selected a categorical Block random-slope model (comparison p=.00177); ln SCR rate retained random intercepts (p=.18261).

Outcome | Effect | F | df1 | df2 | p
SCL | Group | 2.5719 | 1 | 28.807 | .11969
SCL | Block | 12.215 | 2 | 30 | .00013156
SCL | Group x Block | 2.9604 | 2 | 30 | .067083
SCL | Resting SCL baseline | 310.48 | 1 | 29 | 4.9327e-17
ln SCR rate | Group | .92894 | 1 | 28 | .34339
ln SCR rate | Block | .59082 | 2 | 58 | .55717
ln SCR rate | Group x Block | .95463 | 2 | 58 | .39092
ln SCR rate | ln resting SCR baseline | 1.0265 | 1 | 28 | .31965

After adjustment for resting-state SCL, there was a significant main effect of block on mean SCL, F(2, 30)=12.22, p<.001. The group effect, F(1, 28.81)=2.57, p=.120, and group-by-block interaction, F(2, 30)=2.96, p=.067, were not statistically significant. Resting-state SCL was positively associated with training SCL, F(1, 29)=310.48, p<.001. For log-transformed SCR rate, no significant effects of group, block, their interaction, or the baseline covariate were observed (all p>.05).

Final residual skewness was -.197 for SCL and .460 for ln SCR rate. Visual inspection showed tail departures, especially an upper-tail observation for ln SCR rate. These diagnostics do not establish that assumptions are fully met. The categorical random-slope SCL covariance model is complex relative to three observations per participant and warrants sensitivity assessment before publication.

Implementation provenance

Installed environment checked: NeuroKit2 0.2.13; original extraction version was not recorded in historical CSV files. Threshold verified against installed eda_findpeaks and signal_findpeaks source. SCL remains in recorded units; no unverified conversion to microsiemens was applied.
https://neuropsychology.github.io/NeuroKit/_modules/neurokit2/eda/eda_findpeaks.html
https://neuropsychology.github.io/NeuroKit/_modules/neurokit2/signal/signal_findpeaks.html
